# Tutorial 2: Dense BioModels Parameter Sweep

Estimated time: 30-50 minutes

## Prerequisites
`libroadrunner` installed (the BioModels SBML runner). Run `mm doctor` to inspect your env, then `conda install -c conda-forge libroadrunner` if it's missing.

## Learning aims
- Primary package aim: drive a real BioModels SBML model with a DOE plan
- Secondary scientific aim: connect parameter perturbations to model dynamics


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Self-contained: walks up to find the repo root, adds src/ to sys.path,
# then imports the official bootstrap helper for chdir + later run_cli use.
import os, sys
from pathlib import Path

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    _marker = _candidate / 'pyproject.toml'
    if _marker.is_file() and 'name = "metamodeler"' in _marker.read_text():
        ROOT = _candidate
        break
else:
    raise FileNotFoundError('Could not locate metamodeler repo root from ' + str(_here))

_src = str((ROOT / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
if Path.cwd().resolve() != ROOT.resolve():
    os.chdir(ROOT)

from metamodeler.tutorial import bootstrap, run_cli  # noqa: E402
ROOT = bootstrap()
print('Repo root:', ROOT)


## Step 1: Validate the BioModels spec (no SBML execution yet)

In [ ]:
run_cli('validate', 'tutorials/specs/model.biomodels.quick.json')
run_cli('plan', 'tutorials/specs/model.biomodels.quick.json')


## Step 2: Execute runs (requires `libroadrunner`)
If `libroadrunner` isn't installed, the cell below detects it and skips execution
with an actionable message — validation and planning above always work.


In [ ]:
import importlib.util

if importlib.util.find_spec('roadrunner') is None:
    print('Skipping run: libroadrunner is not installed in this env.')
    print("Install with: conda install -c conda-forge libroadrunner")
else:
    run_cli('run', 'tutorials/specs/model.biomodels.quick.json')


## Step 3: Visualize the response across the sweep (if runs exist)

In [ ]:
import json
import matplotlib.pyplot as plt

runs_root = ROOT / 'tmp/tutorials/biomodels_store/runs'
if not runs_root.is_dir() or not any(runs_root.iterdir()):
    print('No runs yet — Step 2 needs libroadrunner.')
else:
    rows = []
    for run_dir in sorted(p for p in runs_root.iterdir() if p.is_dir()):
        try:
            inp = json.loads((run_dir / 'inputs.json').read_text())
            out = json.loads((run_dir / 'outputs.json').read_text())
            rows.append((inp, out))
        except FileNotFoundError:
            continue
    print(f'Loaded {len(rows)} runs.')
    plt.figure(figsize=(6, 4))
    plt.title('BioModels sweep — first numeric output per run')
    for _, out in rows[:20]:
        first_key = next(iter(out))
        v = out[first_key]
        if isinstance(v, dict) and first_key in v:
            v = v[first_key]
        if isinstance(v, list):
            plt.plot(v[:200], alpha=0.5)
    plt.xlabel('time index')
    plt.ylabel('value')
    plt.grid(True, alpha=0.3)
    plt.show()


## Scientific checkpoint
- Are dynamic responses smooth or oscillatory across the sweep?
- What does the spread tell you about parameter sensitivity?
